In [1]:
import re
import sys
import json

# 完整的文件路径
file_path = "/home/guoziyang/output/textbook/计算机算法与分析/MinerU_计算机算法设计与分析（第三版）-王晓东编__20251106070504.md"

# 使用 'r' 模式以文本方式打开文件，并指定编码为 'utf-8' (推荐用于包含中文或特殊字符的文本)
with open(file_path, 'r', encoding='utf-8') as f:
    # 使用 read() 方法读取文件的所有内容
    markdown_content = f.read()

    print("--- 文件内容摘要 (前 400 个字符) ---")
    print(markdown_content[:100])

FileNotFoundError: [Errno 2] No such file or directory: '/home/guoziyang/output/textbook/计算机算法与分析/MinerU_计算机算法设计与分析（第三版）-王晓东编__20251106070504.md'

In [ ]:
import yaml
from camel.models import OpenAIModel

def load_llm_client(config_path):
    """
    根据你的 YAML 加载对应模型的 (api_key, api_base, model)
    并构造 openai SDK 的客户端。
    """
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    llm_use = cfg["llm"]["use"]
    provider_cfg = cfg["llm"][llm_use]

    # 支持 key 池：如果是 list，则取第一个 key（可改成轮询）
    if isinstance(provider_cfg, list):
        provider_cfg = provider_cfg[0]

    model_name = provider_cfg["model"]
    api_key = provider_cfg["api_key"]
    api_base = provider_cfg["api_base"]  # 你 YAML 的字段名

    # Camel-AI 要求 url=xxx
    url = api_base.rstrip("/")  # 去掉结尾斜杠避免重复

    # 构建 model 配置字典（过滤 None）
    model_config = {
        "temperature": provider_cfg.get("temperature"),
        "top_p": provider_cfg.get("top_p"),
        "max_tokens": provider_cfg.get("max_tokens")
    }
    model_config = {k: v for k, v in model_config.items() if v is not None}

    model = OpenAIModel(
        model_type=model_name,       
        model_config_dict=model_config,
        api_key=api_key,
        url=url                     
    )

    return model

In [ ]:
import json
from camel.agents import ChatAgent

def extract_metadata_llm(text, config_path=os.path.join(PROJECT_ROOT, "", "config", "base2.yaml")):
    model = load_llm_client(config_path)

    system_prompt = "你是一个专业的中文图书元信息抽取助手，严格输出 JSON。"
    agent = ChatAgent(system_prompt, model=model)

    user_prompt = f"""
        请从下面内容中提取书籍的元信息，务必输出合法 JSON。

        文本内容（前 2000 字）：
        {text}

        输出格式如下：
        {{
        "title": "",
        "author": "",
        "year": "",
        "extra": ""
        }}
    """

    response = agent.step(user_prompt).msg.content.strip()

    # 去掉可能的 ```json 包裹
    response = response.replace("```json", "").replace("```", "")

    try:
        return json.loads(response)
    except:
        print("⚠️ CAMEL 输出的 JSON 不合法，返回空对象")
        return {}

In [ ]:
metadata = extract_metadata_llm(markdown_content[:2000])
print(metadata)

{'title': '计算机算法设计与分析（第3版）', 'author': '王晓东', 'year': '', 'extra': {'publisher': '電子工業出版社', 'location': '北京', 'description': '普通高等教育“十一五”国家级规划教材，高等学校规划教材'}}


In [ ]:
import re
import sys

def chunk_document(file_path):
    """
    Markdown 文档切分 —— 面向《算法设计与分析》教材的规则定制版

    规则摘要：
    1. “第N章 ...” → 章节根节点，level = 1
    2. 开头数字编号（如 1.2, 3.4.5） → 一般标题，level = 点数 + 1
    3. 含“题”字的标题：
       3.1 无前置数字编号 → 视作当前路径的子标题（level = parent + 1）
       3.2 有前置数字编号：
           - 若路径中存在“习题X” → 作为该“习题X”的子标题，同级分组（如 1. 算法分析题 / 2.算法实现题）
           - 否则 → 退化为普通数字标题
    4. 其他无编号标题 → 视作当前路径的子标题（level = parent + 1）
    5. 标题前面的多余标点（. 、，：等）会自动清理
    """

    # === 1. 读取文件 ===
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
    except Exception as e:
        print(f"文件读取失败: {e}", file=sys.stderr)
        return None

    # === 2. 元数据（可选） ===
    try:
        metadata = extract_metadata_llm("".join(lines[:2000]))
    except Exception:
        metadata = {}

    # === 3. 标题正则 ===
    # group(1): 开头的数字编号（如 "1", "1.2", "3.4.5"）
    # group(3): 其余文本
    heading_pattern = re.compile(r'^#{1,6}\s*(\d+(\.\d+)*)?\s*(.*)')

    ignore_titles = {
        "计算机", "算法设计与分析", "(第3版)",
        "普通高等教育“十一五”国家级规划教材",
        "高等学校规划教材", "目录"
    }

    # === 4. 收集所有标题位置 ===
    split_points = []

    for idx, raw_line in enumerate(lines):
        line = raw_line.strip()
        m = heading_pattern.match(line)
        if not m:
            continue

        numbering = m.group(1)            # 可能为 None
        raw_title_body = m.group(3).strip()

        if raw_title_body in ignore_titles:
            continue

        # 检测“第N章”
        chapter_match = re.search(r'第(\d+)章', raw_title_body)

        # 清理标题前端多余符号（把 ". 算法分析题" 变成 "算法分析题"）
        cleaned_title = re.sub(r'^[\s\.\u3002\uFF0C\uFF1A:，、\-—·]+', '', raw_title_body)

        split_points.append({
            "line_index": idx,
            "numbering": numbering,
            "raw_title": raw_title_body,
            "title": cleaned_title,
            "chapter_match": chapter_match,
        })

    # === 5. 构建 chunks ===
    chunks = []
    logical_path = []   # 栈：[{title, numbering, logical_level}, ...]

    chunk_id = 1

    # 文档开头非标题部分
    if split_points:
        first_idx = split_points[0]["line_index"]
        intro = "".join(lines[:first_idx]).strip()
        if intro:
            chunks.append({
                "chunk_id": chunk_id,
                "title": "文档起始/前言/版权信息",
                "is_numbered": False,
                "logical_level": 0,
                "content": intro,
                "metadata": metadata,
                "path_titles": ["文档起始/前言/版权信息"],
                "path_numbering": ["0"],
            })
            chunk_id += 1

    # === 6. 遍历标题，生成 chunk ===
    for i, sp in enumerate(split_points):
        start = sp["line_index"]
        end = split_points[i + 1]["line_index"] if i + 1 < len(split_points) else len(lines)
        content = "".join(lines[start:end]).strip()

        raw_title = sp["raw_title"]
        title = sp["title"]
        numbering = sp["numbering"]
        chapter_match = sp["chapter_match"]

        has_ti = ("题" in raw_title)
        has_numbering = bool(numbering)

        # ---------- 计算 current_level ----------
        if chapter_match:
            # 章节：第N章
            numbering = chapter_match.group(1)
            current_level = 1

        else:
            # 查找最近的“习题X”所在层级
            exercise_level = None
            for node in reversed(logical_path):
                if "习题" in node["title"]:
                    exercise_level = node["logical_level"]
                    break

            # 1）题字标题 + 无数字编号 → 子标题（往下钻）
            if has_ti and not has_numbering:
                if logical_path:
                    current_level = logical_path[-1]["logical_level"] + 1
                else:
                    current_level = 1

            # 2）题字标题 + 有数字编号 → 习题块中的分组标题
            elif has_ti and has_numbering:
                if exercise_level is not None:
                    # 作为“习题X”的直接子标题（同级分组）
                    current_level = exercise_level + 1
                else:
                    # 不在习题块里，就按普通数字标题处理
                    current_level = numbering.count('.') + 1

            # 3）普通数字标题
            elif has_numbering:
                current_level = numbering.count('.') + 1

            # 4）普通无编号标题 → 子标题
            else:
                if logical_path:
                    current_level = logical_path[-1]["logical_level"] + 1
                else:
                    current_level = 1

        # ---------- 更新 logical_path ----------
        # 正确维护路径，只保留真正的父节点链
        while logical_path and logical_path[-1]["logical_level"] >= current_level:
            logical_path.pop()

        logical_path.append({
            "title": title,
            "numbering": numbering,
            "logical_level": current_level,
        })


        # ---------- 路径字段 ----------
        path_titles = [n["title"] for n in logical_path]
        path_numbering = [
            (n["numbering"] if n["numbering"] else n["title"])
            for n in logical_path
        ]

        # ---------- 构建 chunk ----------
        chunk = {
            "chunk_id": chunk_id,
            "title": title,
            "is_numbered": numbering is not None,
            "logical_level": current_level,
            "content": content,
            "metadata": metadata.copy(),
            "path_titles": path_titles,
            "path_numbering": path_numbering,
        }
        if numbering:
            chunk["metadata"]["chapter_numbering"] = numbering

        chunks.append(chunk)
        chunk_id += 1

    return chunks


In [ ]:
output_path = "document_chunks.json"
chunks_list = chunk_document(file_path)

if chunks_list:
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(chunks_list, f, ensure_ascii=False, indent=4)
        print(f"成功切分文档并保存到: {output_path}")
        
        # 打印前5个Chunk的摘要信息，以便用户快速查看结果
        print("\n--- 前5个Chunk摘要 ---")
        for chunk in chunks_list[:5]:
            print(f"ID: {chunk['chunk_id']}, Title: {chunk['title']}")
            print(f"  Content Snippet: {chunk['content'][:100]}...")
            print(f"  Metadata: {chunk['metadata']}")
            print("-" * 20)
            
    except Exception as e:
        print(f"保存JSON文件时发生错误: {e}", file=sys.stderr)

成功切分文档并保存到: document_chunks.json

--- 前5个Chunk摘要 ---
ID: 1, Title: 文档起始/前言/版权信息
  Content Snippet: # 计算机

# 算法设计与分析

# (第3版)

王晓东 编著

计算机学科教学计划

# 普通高等教育“十一五”国家级规划教材

# 高等学校规划教材...
  Metadata: {'title': '计算机算法设计与分析', 'author': '王晓东', 'year': '第3版', 'extra': {'publisher': '電子工業出版社', 'location': '北京', 'type': ['普通高等教育“十一五”国家级规划教材', '高等学校规划教材']}}
--------------------
ID: 2, Title: 计算机算法设计与分析
  Content Snippet: # 计算机算法设计与分析

（第3版）

王晓东 编著

電子工業出版社

Publishing House of Electronics Industry

北京·BEIJING...
  Metadata: {'title': '计算机算法设计与分析', 'author': '王晓东', 'year': '第3版', 'extra': {'publisher': '電子工業出版社', 'location': '北京', 'type': ['普通高等教育“十一五”国家级规划教材', '高等学校规划教材']}}
--------------------
ID: 3, Title: 前言
  Content Snippet: # 前言

计算机的普及极大地改变了人们的生活。目前，各行业、各领域都广泛采用了计算机信息技术，并由此产生出开发各种应用软件的需求。为了以最小的成本、最快的速度、最好的质量开发出适合各种应用需求的软件...
  Metadata: {'title': '计算机算法设计与分析', 'author': '王晓东', 'year': '第3版', 'extra': {'publisher': '電子工業出版社', 'location': '北京', 'type': ['普通高等教育“十一五”国家级规划教材', '高等学校规划教材']}}
----